In [0]:
from pyspark.sql.functions import col, current_timestamp, expr, to_utc_timestamp, coalesce, lit, row_number
from pyspark.sql.window import Window

# 1. Define Unity Catalog Paths
catalog = "training_medallion"
schema_bronze = "01_bronze"
schema_silver = "02_silver"
schema_gold = "03_gold"

base_volume = f"/Volumes/{catalog}/00_landing/raw_landing_zone"
landing_orders_path = f"{base_volume}/orders_landing"
silver_checkpoint = f"{base_volume}/_checkpoints/orders_silver"

# 2. Ensure catalog, schemas, and landing directories exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_bronze}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_silver}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_gold}")

dbutils.fs.mkdirs(landing_orders_path)

# 3. Create mock Bronze orders Delta table
bronze_orders_data = [
    ("ORD-1001", "CUST-501", "MERCH-801", 150.00, "2026-10-18 10:00:00", "EST", "COMPLETED"),
    ("ORD-1001", "CUST-501", "MERCH-801", 150.00, "2026-10-18 10:00:05", "EST", "COMPLETED"), # Duplicate webhook retry
    ("ORD-1002", "CUST-502", "MERCH-802", None,   "2026-10-18 11:15:00", "UTC", "PENDING"),   # Null order_amount
    ("ORD-1003", "CUST-503", "MERCH-801", -50.00, "2026-10-18 12:30:00", "PST", "FAILED"),    # Invalid amount
    ("ORD-1004", "CUST-504", "MERCH-803", 499.99, "2026-10-18 14:00:00", "UTC", "COMPLETED")
]

columns = ["order_id", "customer_id", "merchant_id", "order_amount", "order_timestamp", "timezone", "status"]
df_raw = spark.createDataFrame(bronze_orders_data, columns)\
    .withColumn("ingestion_time", current_timestamp())\
    .withColumn("source_file", lit("s3://landing/orders_batch_1.json"))

df_raw.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema_bronze}.raw_orders")

# 4. Generate JSON landing file for Auto Loader / DLT Pipeline testing
raw_orders_payload_1 = """{"order_id": "ORD-5001", "customer_id": "CUST-101", "merchant_id": "MERCH-801", "order_amount": 250.00, "order_timestamp": "2026-10-18 10:00:00", "status": "COMPLETED"}
{"order_id": "ORD-5002", "customer_id": "CUST-102", "merchant_id": "MERCH-802", "order_amount": -15.00, "order_timestamp": "2026-10-18 10:05:00", "status": "COMPLETED"}
{"order_id": "ORD-5003", "customer_id": "CUST-103", "merchant_id": "MERCH-801", "order_amount": 120.50, "order_timestamp": "2026-10-18 10:10:00", "status": "PENDING"}"""

dbutils.fs.put(f"{landing_orders_path}/orders_batch_1.json", raw_orders_payload_1, overwrite=True)

print(f"✅ Raw Bronze table & landing JSON file created successfully in:\n{landing_orders_path}")

Wrote 504 bytes.
✅ Raw Bronze table & landing JSON file created successfully in:
/Volumes/training_medallion/00_landing/raw_landing_zone/orders_landing


In [0]:
%sql
select * from training_medallion.01_bronze.raw_orders

order_id,customer_id,merchant_id,order_amount,order_timestamp,timezone,status,ingestion_time,source_file
ORD-1001,CUST-501,MERCH-801,150.0,2026-10-18 10:00:00,EST,COMPLETED,2026-10-18T16:14:25.761Z,s3://landing/orders_batch_1.json
ORD-1001,CUST-501,MERCH-801,150.0,2026-10-18 10:00:05,EST,COMPLETED,2026-10-18T16:14:25.761Z,s3://landing/orders_batch_1.json
ORD-1002,CUST-502,MERCH-802,null,2026-10-18 11:15:00,UTC,PENDING,2026-10-18T16:14:25.761Z,s3://landing/orders_batch_1.json
ORD-1003,CUST-503,MERCH-801,-50.0,2026-10-18 12:30:00,PST,FAILED,2026-10-18T16:14:25.761Z,s3://landing/orders_batch_1.json
ORD-1004,CUST-504,MERCH-803,499.99,2026-10-18 14:00:00,UTC,COMPLETED,2026-10-18T16:14:25.761Z,s3://landing/orders_batch_1.json


In [0]:
from pyspark.sql.functions import col, current_timestamp, expr, to_utc_timestamp, coalesce, lit, row_number
from pyspark.sql.window import Window

# 1. Read from Bronze layer (Fixed: spark.table)
df_bronze = spark.table(f"{catalog}.{schema_bronze}.raw_orders")

# 2. Normalize Timestamp to UTC & Handle Null Amounts
df_normalized = df_bronze \
    .withColumn(
        "order_timestamp_utc",
        expr("""
            CASE 
                WHEN timezone = 'EST' THEN to_utc_timestamp(order_timestamp, 'America/New_York')
                WHEN timezone = 'PST' THEN to_utc_timestamp(order_timestamp, 'America/Los_Angeles')
                ELSE to_utc_timestamp(order_timestamp, 'UTC')
            END
        """)
    ) \
    .withColumn("order_amount_cleaned", coalesce(col("order_amount"), lit(0.00)))

# 3. Deduplicate retried transactions using Window functions
window_spec = Window.partitionBy("order_id").orderBy(col("order_timestamp_utc").desc())

df_silver_clean = df_normalized \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num", "order_amount") \
    .withColumnRenamed("order_amount_cleaned", "order_amount")

# 4. Write cleaned dataset to Silver Delta Table
df_silver_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{schema_silver}.clean_orders")

print("Silver transformation completed successfully.")

Silver transformation completed successfully.


In [0]:
%sql
select * from training_medallion.02_silver.clean_orders

order_id,customer_id,merchant_id,order_timestamp,timezone,status,ingestion_time,source_file,order_timestamp_utc,order_amount
ORD-1001,CUST-501,MERCH-801,2026-10-18 10:00:05,EST,COMPLETED,2026-10-18T16:14:25.761Z,s3://landing/orders_batch_1.json,2026-10-18T14:00:05.000Z,150.0
ORD-1002,CUST-502,MERCH-802,2026-10-18 11:15:00,UTC,PENDING,2026-10-18T16:14:25.761Z,s3://landing/orders_batch_1.json,2026-10-18T11:15:00.000Z,0.0
ORD-1003,CUST-503,MERCH-801,2026-10-18 12:30:00,PST,FAILED,2026-10-18T16:14:25.761Z,s3://landing/orders_batch_1.json,2026-10-18T19:30:00.000Z,-50.0
ORD-1004,CUST-504,MERCH-803,2026-10-18 14:00:00,UTC,COMPLETED,2026-10-18T16:14:25.761Z,s3://landing/orders_batch_1.json,2026-10-18T14:00:00.000Z,499.99


In [0]:
import dlt
from pyspark.sql.functions import col, coalesce, lit, expr, to_utc_timestamp, row_number
from pyspark.sql.window import Window

# 1. Ingest raw Bronze stream in DLT
@dlt.table(
    name="dlt_bronze_orders",
    comment="Raw streaming e-commerce orders landed from cloud storage."
)
def dlt_bronze_orders():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .load("/Volumes/training_medallion/00_landing/raw_landing_zone/orders_landing")
    )

# 2. Transform to Silver with Data Quality Expectations
@dlt.table(
    name="dlt_silver_orders",
    comment="Cleaned and validated orders dataset."
)
@dlt.expect("valid_order_id", "order_id IS NOT NULL")
@dlt.expect_or_drop("positive_order_amount", "order_amount > 0") # Drops negative/invalid amounts
@dlt.expect_or_fail("valid_status", "status IN ('COMPLETED', 'PENDING', 'FAILED')") # Aborts if unknown status appears
def dlt_silver_orders():
    df_raw = dlt.read_stream("dlt_bronze_orders")
    
    # Deduplication and Normalization Logic
    return (
        df_raw
        .filter(col("order_id").isNotNull())
        .withColumn("order_amount", coalesce(col("order_amount"), lit(0.00)))
        .withColumn("order_timestamp_utc", to_utc_timestamp(col("order_timestamp"), "UTC"))
    )

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-7633642379212989>, line 1
----> 1 import dlt
      2 from pyspark.sql.functions import col, coalesce, lit, expr, to_utc_timestamp, row_number
      3 from pyspark.sql.window import Window

ModuleNotFoundError: No module named 'dlt'

In [0]:
from pyspark.sql.functions import col, current_timestamp, expr, to_utc_timestamp, coalesce, lit, row_number,to_date, sum, count, round
from pyspark.sql.window import Window

# Define Unity Catalog Paths
catalog = "training_medallion"
schema_bronze = "01_bronze"
schema_silver = "02_silver"
schema_gold = "03_gold"
# 1. Read clean Silver table
df_silver = spark.table(f"{catalog}.{schema_silver}.clean_orders") 

# 2. Aggregate metrics by Merchant and Date
df_gold_merchant_daily = df_silver \
    .filter(col("status") == "COMPLETED") \
    .withColumn("order_date", to_date(col("order_timestamp_utc"))) \
    .groupBy("order_date", "merchant_id") \
    .agg(
        round(sum("order_amount"), 2).alias("total_gmv"),
        count("order_id").alias("completed_orders")
    )

# 3. Save to Gold Delta Table
df_gold_merchant_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{schema_gold}.daily_merchant_sales")

print("Gold business table created successfully.")

Gold business table created successfully.


In [0]:
%sql
SELECT 
  order_date, 
  merchant_id, 
  total_gmv, 
  completed_orders 
FROM training_medallion.03_gold.daily_merchant_sales
ORDER BY order_date DESC, total_gmv DESC;

order_date,merchant_id,total_gmv,completed_orders
2026-10-18,MERCH-803,499.99,1
2026-10-18,MERCH-801,150.0,1
